# FHVHV 2024–2025 exploration: memory usage in pandas

Goal: measure how much RAM one month of FHVHV takes, find out which columns are the heaviest, shrink them with smaller dtypes, and project the result to 2024 + 2025.

## 1. Imports and setup

`__file__` does not exist in Jupyter. The notebook runs from its own folder, so `Path("datasets")` already points to the right place.

In [ ]:
import gc
import time

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

data_dir = Path("datasets")
file_path = data_dir / "fhvhv_tripdata_2025-01.parquet"       # reference month
file_path_2024 = data_dir / "fhvhv_tripdata_2024-01.parquet"  # used in section 11

print("pandas", pd.__version__)
print("Files exist?", file_path.exists(), file_path_2024.exists())

## 2. Inventory of all files (metadata only)

Row counts come from the Parquet metadata, so nothing is loaded into RAM.

In [ ]:
records = []
for f in sorted(data_dir.glob("*_tripdata_*.parquet")):
    dataset, _, year_month = f.stem.partition("_tripdata_")
    records.append({
        "dataset": dataset,
        "year": int(year_month[:4]),
        "month": year_month,
        "rows": pq.ParquetFile(f).metadata.num_rows,
        "MB": f.stat().st_size / 1e6,
    })
inventory = pd.DataFrame(records)

(inventory.groupby(["dataset", "year"])
    .agg(files=("month", "count"), rows=("rows", "sum"), MB=("MB", "sum"))
    .assign(rows_M=lambda x: (x.pop("rows") / 1e6).round(1), MB=lambda x: x["MB"].round(0)))

## 3. Schema and schema drift (without loading data)

Reads only the Parquet headers. Then compares every FHVHV file against the reference month to find missing columns, extra columns or type changes.

In [ ]:
print(pq.read_schema(file_path))

In [ ]:
def schema_of(path: Path) -> dict:
    return {field.name: str(field.type) for field in pq.read_schema(path)}

reference = schema_of(file_path)
fhvhv_files = sorted(data_dir.glob("fhvhv_tripdata_*.parquet"))

matching = 0
for f in fhvhv_files:
    cols = schema_of(f)
    missing = sorted(set(reference) - set(cols))
    extra = sorted(set(cols) - set(reference))
    changed = {c: (cols[c], reference[c]) for c in set(cols) & set(reference) if cols[c] != reference[c]}
    if missing or extra or changed:
        print(f"{f.stem}: missing={missing} extra={extra} type changes={changed}")
    else:
        matching += 1

print(f"\n{matching} of {len(fhvhv_files)} files match the reference schema exactly")

## 4. Compare backends (optional)

Loads the same month with each backend, measures RAM and time, then deletes it. This is the slowest cell. You can skip it after running it once.

In [ ]:
for backend in ["numpy_nullable", "pyarrow", None]:
    start = time.perf_counter()
    if backend is None:
        tmp = pd.read_parquet(file_path)  # pandas default
    else:
        tmp = pd.read_parquet(file_path, dtype_backend=backend)
    gb = tmp.memory_usage(deep=True).sum() / 1e9
    name = backend or "default"
    print(f"{name:15} {gb:6.2f} GB in RAM   {time.perf_counter() - start:5.1f}s")
    del tmp

gc.collect();

## 5. Load one month

From here on, `df` stays in the kernel's memory and the next cells reuse it.

In [ ]:
df = pd.read_parquet(file_path)
n_rows = len(df)
print(f"{n_rows:,} rows x {df.shape[1]} columns")
df.head()

## 6. Memory per column (MB)

In [ ]:
mem = df.memory_usage(deep=True).drop("Index")
print(f"Total: {mem.sum() / 1e9:.2f} GB")
(mem / 1e6).sort_values(ascending=False).round(1).to_frame("MB")

## 7. Distinct values in the text columns

`category` only pays off when there are few distinct values.

In [ ]:
TEXT_COLS = ["hvfhs_license_num", "dispatching_base_num", "originating_base_num",
             "shared_request_flag", "shared_match_flag", "access_a_ride_flag",
             "wav_request_flag", "wav_match_flag"]
FLOAT_COLS = ["trip_miles", "base_passenger_fare", "tolls", "bcf", "sales_tax",
              "congestion_surcharge", "airport_fee", "tips", "driver_pay",
              "cbd_congestion_fee"]  # cbd_congestion_fee only exists from 2025 on

df[TEXT_COLS].nunique().to_frame("distinct values")

## 8. Optimize dtypes and compare

`optimize()` only converts the columns that exist in the DataFrame, so it also works for 2024 files, which do not have `cbd_congestion_fee`.

In [ ]:
def optimize(df: pd.DataFrame) -> pd.DataFrame:
    text_cols = [c for c in TEXT_COLS if c in df.columns]
    float_cols = [c for c in FLOAT_COLS if c in df.columns]

    df[text_cols] = df[text_cols].astype("category")
    df[float_cols] = df[float_cols].astype("float32")
    df["trip_time"] = df["trip_time"].astype("int32")
    df[["PULocationID", "DOLocationID"]] = df[["PULocationID", "DOLocationID"]].astype("int16")
    return df

before = df.memory_usage(deep=True).sum() / 1e9
df = optimize(df)
after = df.memory_usage(deep=True).sum() / 1e9

print(f"Before: {before:.2f} GB   After: {after:.2f} GB   ({before / after:.1f}x smaller)")

In [ ]:
(df.memory_usage(deep=True).drop("Index") / 1e6).sort_values(ascending=False).round(1).to_frame("MB after")

## 9. FHVHV projection for 2024 and 2025

Bytes per row measured on the reference month × total rows of each year (from the inventory). 2024 has one column fewer, so its numbers are slightly overestimated (~4%).

In [ ]:
raw_bytes_per_row = before * 1e9 / n_rows
opt_bytes_per_row = after * 1e9 / n_rows

fhvhv_rows = inventory[inventory["dataset"] == "fhvhv"].groupby("year")["rows"].sum()
projection = pd.DataFrame({
    "rows (M)": fhvhv_rows / 1e6,
    "raw GB": fhvhv_rows * raw_bytes_per_row / 1e9,
    "optimized GB": fhvhv_rows * opt_bytes_per_row / 1e9,
})
projection.loc["total"] = projection.sum()

print(f"{raw_bytes_per_row:.0f} bytes/row raw, {opt_bytes_per_row:.0f} bytes/row optimized")
projection.round(1)

## 10. Free memory

Run this before loading other months. If RAM stays high, use **Restart Kernel**.

In [ ]:
del df
gc.collect();

## 11. Check a 2024 month

Confirms that `optimize()` works without `cbd_congestion_fee` and that a 2024 month shrinks by about the same factor.

In [ ]:
df_2024 = pd.read_parquet(file_path_2024)
print(f"{len(df_2024):,} rows x {df_2024.shape[1]} columns")
print("Columns missing compared with 2025:", sorted(set(reference) - set(df_2024.columns)))

before_2024 = df_2024.memory_usage(deep=True).sum() / 1e9
df_2024 = optimize(df_2024)
after_2024 = df_2024.memory_usage(deep=True).sum() / 1e9
print(f"Before: {before_2024:.2f} GB   After: {after_2024:.2f} GB   ({before_2024 / after_2024:.1f}x smaller)")

del df_2024
gc.collect();

## 12. Projection for all datasets (raw, not optimized)

For each dataset: load January 2025, measure bytes per row, and multiply by the total rows of each year.
The four datasets have different columns, so in practice they are analyzed separately, never in one DataFrame.

In [ ]:
results = []
for dataset in ["fhvhv", "yellow", "fhv", "green"]:
    sample = pd.read_parquet(data_dir / f"{dataset}_tripdata_2025-01.parquet")
    bytes_per_row = sample.memory_usage(deep=True).sum() / len(sample)
    del sample

    rows_per_year = inventory[inventory["dataset"] == dataset].groupby("year")["rows"].sum()
    for year, rows in rows_per_year.items():
        results.append({
            "dataset": dataset,
            "year": year,
            "rows (M)": rows / 1e6,
            "raw GB (est.)": rows * bytes_per_row / 1e9,
        })

gc.collect()
all_projection = pd.DataFrame(results).set_index(["dataset", "year"])
print(f"Total for all datasets: {all_projection['raw GB (est.)'].sum():.1f} GB")
all_projection.round(1)